# NSE Stock Direction Predictor — End-to-End Workflow

This notebook reproduces the project workflow from the exported five-minute NSE OHLCV CSV: EDA, cleaning, visualisation, feature engineering, labels, training, evaluation, news sentiment, prediction, testing, and saving the LightGBM `.pkl` model.

It calls the same functions in `src/` used by the application, so the notebook remains aligned with the deployed pipeline.

## 1. Setup

Run this notebook with the project's Python environment selected.

In [ ]:
from pathlib import Path
import os, sys, subprocess

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CSV_PATH = ROOT / 'data' / 'nse_5min_ohlcv.csv'
assert CSV_PATH.exists(), f'Missing {CSV_PATH}. Export data/market.db first.'
print(f'Project root: {ROOT}')
print(f'Dataset: {CSV_PATH}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from config import INDEX_TICKER, HORIZON_BARS, MODEL_PATH, METADATA_PATH
from src.features import build_features, get_feature_columns
from src.labels import add_labels_to_features
from src.train import split_held_out, time_split, train_model, save_model
from src.evaluate import evaluate_split, print_report, save_results, write_summary

sns.set_theme(style='darkgrid', context='notebook')
pd.set_option('display.max_columns', 50)

## 2. Load and audit the dataset

In [ ]:
raw = pd.read_csv(CSV_PATH)
raw['ts'] = pd.to_datetime(raw['ts'])
display(raw.head())
print(f'Rows: {len(raw):,}')
print(f'Tickers: {raw.ticker.nunique()}')
print(f'Date range: {raw.ts.min()} to {raw.ts.max()}')
display(raw.groupby('ticker').agg(rows=('ticker', 'size'), start=('ts', 'min'), end=('ts', 'max')).sort_index())
display(raw.describe(include='all'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
raw.groupby('ticker').size().sort_values().plot.bar(ax=axes[0], color='#22c55e')
axes[0].set_title('Five-minute bars per ticker'); axes[0].set_ylabel('rows')
sample = raw[raw.ticker.eq('RELIANCE.NS')]
axes[1].plot(sample.ts, sample.close, color='#3b82f6', linewidth=1)
axes[1].set_title('RELIANCE.NS close price'); axes[1].set_xlabel('timestamp'); axes[1].set_ylabel('INR')
plt.tight_layout()

## 3. Clean and validate OHLCV bars

In [ ]:
required = ['ticker', 'ts', 'open', 'high', 'low', 'close', 'volume']
assert set(required).issubset(raw.columns)

clean = raw[required].copy().dropna(subset=['ticker', 'ts', 'open', 'high', 'low', 'close'])
clean = clean.drop_duplicates(['ticker', 'ts']).sort_values(['ticker', 'ts'])
clean = clean[(clean.close > 0) & (clean.open > 0) & (clean.high >= clean.low) & (clean.volume >= 0)]
assert clean[['open', 'high', 'low', 'close', 'volume']].isna().sum().sum() == 0
assert not clean.duplicated(['ticker', 'ts']).any()
clean.to_csv(ROOT / 'data' / 'nse_5min_ohlcv_clean.csv', index=False)
print(f'Clean rows: {len(clean):,}; removed: {len(raw) - len(clean):,}')
clean.head()

## 4. Feature engineering and labels

The production feature function uses only information available at each bar. Labels are the next three five-minute bars (15 minutes) with a volatility-adaptive neutral band; labels spanning market days are dropped.

In [ ]:
def indexed_ohlcv(frame):
    return frame.drop(columns='ticker', errors='ignore').set_index('ts').sort_index()

nifty_df = indexed_ohlcv(clean[clean.ticker.eq(INDEX_TICKER)])
stock_tickers = sorted(set(clean.ticker) - {INDEX_TICKER})

def prepare_tickers(tickers):
    frames = []
    for ticker in tickers:
        ohlcv = indexed_ohlcv(clean[clean.ticker.eq(ticker)])
        features = build_features(ohlcv, nifty_df=nifty_df, drop_nans=True)
        labelled = add_labels_to_features(features, ohlcv)
        labelled['ticker'] = ticker
        frames.append(labelled)
    return pd.concat(frames).sort_index()

train_tickers, heldout_tickers = split_held_out(stock_tickers)
train_data = prepare_tickers(train_tickers)
heldout_data = prepare_tickers(heldout_tickers)
feature_cols = get_feature_columns()
print('Training tickers:', train_tickers)
print('Held-out tickers:', heldout_tickers)
print('Features:', len(feature_cols))
display(train_data[feature_cols + ['label', 'fwd_ret', 'ticker']].head())

In [ ]:
label_counts = train_data.label.value_counts().reindex(['DOWN', 'NEUTRAL', 'UP'])
label_counts.plot.bar(color=['#ef4444', '#a1a1aa', '#22c55e'], figsize=(7, 4), title=f'15-minute labels ({HORIZON_BARS} bars)')
plt.ylabel('rows'); plt.show()
display(train_data[feature_cols].describe().T[['mean', 'std', 'min', 'max']])

## 5. Feature selection and time-aware splitting

The selected columns are defined in `src.features.get_feature_columns()`. The training split has purge gaps, preventing overlap between forward-looking labels across train/validation/test boundaries.

In [ ]:
train_df, val_df, test_df = time_split(train_data)
display(pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'rows': [len(train_df), len(val_df), len(test_df)],
    'start': [x.index.min() for x in [train_df, val_df, test_df]],
    'end': [x.index.max() for x in [train_df, val_df, test_df]],
}))
feature_cols

## 6. Train, evaluate, and save the model

In [ ]:
model = train_model(train_df, val_df, feature_cols)
test_report = evaluate_split(model, test_df, feature_cols, split_name='notebook_seen')

test_start, test_end = test_df.index.min(), test_df.index.max()
heldout_test = heldout_data[(heldout_data.index >= test_start) & (heldout_data.index <= test_end)]
heldout_report = evaluate_split(model, heldout_test, feature_cols, split_name='notebook_held_out')

print_report(test_report, 'Seen-ticker test')
print_report(heldout_report, 'Held-out-ticker test')
save_results(test_report, 'notebook_seen')
save_results(heldout_report, 'notebook_held_out')
write_summary([test_report, heldout_report])

metrics = {'seen_accuracy': test_report.get('accuracy'), 'seen_macro_f1': test_report.get('macro_f1'), 'held_accuracy': heldout_report.get('accuracy')}
save_model(model, feature_cols, train_df, metrics)
print(f'Model saved: {MODEL_PATH}')
print(f'Metadata saved: {METADATA_PATH}')

In [ ]:
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values()
importance.tail(15).plot.barh(figsize=(8, 6), color='#22c55e', title='LightGBM feature importance')
plt.xlabel('split importance'); plt.show()
pd.DataFrame([{'split': 'seen', **test_report}, {'split': 'held-out', **heldout_report}]).T.head(15)

## 7. News sentiment (optional network step)

Set `FETCH_LIVE_NEWS` to `True` to fetch Google News RSS headlines. FinBERT is used when available; the project falls back to a financial lexicon if necessary.

In [ ]:
FETCH_LIVE_NEWS = False
news_ticker = 'RELIANCE.NS'
if FETCH_LIVE_NEWS:
    from src.news import fetch_headlines
    from src.sentiment import score_headlines, aggregate_sentiment
    fetch_headlines(news_ticker, sleep_sec=0.1)
    score_headlines(batch_size=16)
    display(pd.DataFrame(aggregate_sentiment(news_ticker).get('headlines', [])))
else:
    print('Skipped live news. Set FETCH_LIVE_NEWS = True to run.')

## 8. Replay prediction using the saved `.pkl` model

In [ ]:
from src.predict import predict_for

prediction_ticker = 'RELIANCE.NS'
as_of = clean.loc[clean.ticker.eq(prediction_ticker), 'ts'].max()
prediction = predict_for(prediction_ticker, as_of=as_of, mode='replay', include_news=False)
prediction

## 9. Run the project tests

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests', '-q', '--basetemp', str(ROOT / '.pytest-notebook-tmp')],
    text=True, capture_output=True,
)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError(f'pytest failed with exit code {result.returncode}')

## Outputs

- Clean CSV: `data/nse_5min_ohlcv_clean.csv`
- Model: `models/lgbm_model.pkl`
- Metadata: `models/metadata.json`
- Evaluation reports: `results/notebook_seen_report.json` and `results/notebook_held_out_report.json`

The original Python modules remain the source of truth; this notebook provides their complete, reproducible walkthrough.